# Tech Jobs Analyzer — Full Project Launcher

Runs the full project in one Google Colab session:

**Stage 1 → ChromaDB → Stage 2 → Gemini → Streamlit**

Required Colab Secrets:
- `GEMINI_API_KEY`
- `NGROK_AUTH_TOKEN`

In [ ]:
# Install project dependencies
!pip install -q pandas numpy tqdm chromadb langchain-text-splitters google-genai streamlit python-dotenv pyngrok

from google.colab import drive, userdata
from pathlib import Path
import os

drive.mount("/content/drive")

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is missing. Add it in Colab Secrets.")

if not NGROK_AUTH_TOKEN:
    raise ValueError("NGROK_AUTH_TOKEN is missing. Add it in Colab Secrets.")

DATASET_PATH = Path("/content/drive/MyDrive/postings.csv")
CHROMA_DB_PATH = Path("/content/drive/MyDrive/chroma_db")
COLLECTION_NAME = "tech_jobs"

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["CHROMA_DB_PATH"] = str(CHROMA_DB_PATH)

## Stage 1 — Build or reuse the ChromaDB index

Set `REBUILD_INDEX = True` only when the existing index must be rebuilt.

In [ ]:
import uuid
import pandas as pd
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils import embedding_functions

MAX_ROWS = 10_000
BATCH_SIZE = 100
REBUILD_INDEX = False

embedding_fn = embedding_functions.DefaultEmbeddingFunction()
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))

if REBUILD_INDEX:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

if collection.count() == 0:
    df = pd.read_csv(DATASET_PATH).head(MAX_ROWS)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=40,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    def clean_text(text):
        if pd.isna(text):
            return ""
        return str(text).replace("\n", " ").strip()

    documents = []
    metadatas = []
    ids = []

    for _, row in df.iterrows():
        description = clean_text(row.get("description", ""))
        if not description:
            continue

        job_title = (
            str(row.get("title"))
            if not pd.isna(row.get("title"))
            else "Unknown Title"
        )
        company = (
            str(row.get("company_name"))
            if not pd.isna(row.get("company_name"))
            else "Unknown Company"
        )

        parent_id = f"job_{uuid.uuid4().hex[:8]}"
        chunks = splitter.split_text(description)

        for chunk_index, chunk in enumerate(chunks):
            documents.append(chunk)
            metadatas.append({
                "parent_id": parent_id,
                "job_title": job_title,
                "company": company,
                "chunk_index": chunk_index,
                "total_chunks": len(chunks),
            })
            ids.append(f"{parent_id}_chunk_{chunk_index}")

            if len(documents) >= BATCH_SIZE:
                collection.add(
                    documents=documents,
                    metadatas=metadatas,
                    ids=ids,
                )
                documents.clear()
                metadatas.clear()
                ids.clear()

    if documents:
        collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids,
        )

    print(f"✅ Stage 1 complete — {collection.count():,} chunks indexed.")
else:
    print(f"✅ Stage 1 reused existing index — {collection.count():,} chunks.")

## Stage 2 — Semantic search + grounded Gemini response

In [ ]:
from google import genai
from google.genai import types

TOP_K = 3
MAX_DISTANCE = 0.8

CANDIDATE_MODELS = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.5-flash",
]

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

def build_prompt(query, results):
    context_blocks = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        if distance <= MAX_DISTANCE:
            context_blocks.append(
                f"- Job title: {metadata.get('job_title')}\n"
                f"  Company: {metadata.get('company')}\n"
                f"  Details: {document}"
            )

    context = "\n\n".join(context_blocks)
    if not context:
        context = "No sufficiently relevant jobs were found in the current database."

    return f"""
You are an expert AI Career Advisor analyzing tech job market data.

Answer the user's question strictly from the provided Job Context.
Do not invent jobs, companies, requirements, locations, salaries, or other facts.
If the context does not contain enough information, clearly say so.

### Job Context
{context}

### User Query
{query}

### Answer
"""

def generate_answer(prompt):
    last_error = None

    for model in CANDIDATE_MODELS:
        try:
            response = gemini_client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.1),
            )
            if response.text:
                return response.text
        except Exception as exc:
            last_error = exc

    raise RuntimeError("All configured Gemini models failed.") from last_error

test_query = "Python developer with machine learning experience"

search_results = collection.query(
    query_texts=[test_query],
    n_results=TOP_K,
)

answer = generate_answer(
    build_prompt(test_query, search_results)
)

print("✅ Stage 2 complete.")
print(answer)

## Streamlit Interface

Downloads the current `app.py` from GitHub, starts Streamlit, and creates a temporary ngrok tunnel.

In [ ]:
import subprocess
import time
from pyngrok import ngrok

GITHUB_APP_URL = (
    "https://raw.githubusercontent.com/"
    "rashedkarnoub661-cyber/"
    "tech-jobs-analyzer/"
    "main/app.py"
)

subprocess.run(
    "pkill -f 'streamlit run /content/app.py'",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
subprocess.run(
    "fuser -k 8501/tcp",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

try:
    ngrok.kill()
except Exception:
    pass

subprocess.run(
    ["wget", "-q", "-O", "/content/app.py", GITHUB_APP_URL],
    check=True,
)

subprocess.run(
    ["python", "-m", "py_compile", "/content/app.py"],
    check=True,
)

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

time.sleep(5)

if process.poll() is not None:
    raise RuntimeError("Streamlit failed to start.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8501)

print("✅ Tech Jobs Analyzer is running.")
print(f"🌐 Open the app: {public_url}")